In [ ]:
%%bash

pip install --upgrade 'scikit-learn<1.6.0'

In [ ]:
import numpy as np
import pandas as pd

# 1. Frame the Problem (15 pts)

Answer the following questions.

1. Define the objective: what are you trying to predict?
**Ans: The objective is to predict the SalePrice which is our target variable for each house based on its features.**
2. How should you frame this problem as a machine learning problem?
**Ans: This problem should be framed as a supervised, multiple regression task. This also falls under the category of batch learning.**
3. How should performance be measured? 
**Ans: Performance should be measured using the Root Mean Squared Error (RMSE) between the logarithm of the predicted value and the logarithm of the observed sales price.**

# 2. Get the Data (15 pts)

Minimum expectations:

1. Load the data.
2. Check the size and type of data.

In [ ]:
# Path to the competition data
data_path = "../input/house-prices-advanced-regression-techniques/"

# Load the datasets
train_df = pd.read_csv(data_path + "train.csv")
test_df = pd.read_csv(data_path + "test.csv")

print("Datasets loaded successfully!")


In [ ]:
# Checking size and type of data
print("Training data shape:", train_df.shape)
print("Test data shape:", test_df.shape)

import os
# Get file sizes in KB

# Define full file paths
train_path = data_path + "train.csv"
test_path = data_path + "test.csv"

train_size_kb = os.path.getsize(train_path) / 1024
test_size_kb = os.path.getsize(test_path) / 1024

print(f"Train.csv file size: {train_size_kb:.2f} KB")
print(f"Test.csv file size: {test_size_kb:.2f} KB")

print(train_df.dtypes)
train_df.dtypes.value_counts()

# 3. Explore the Data (15 pts)

General expectations

1. Discuss the types of features (categorical, ordinal, int/float, bounded/unbounded, text, etc.)
2. What features have missing values?
3. Are there any outliers? If so, how can you tell?
4. What features appear usefulness for the task? Are there any that appear useless? Why?
5. Identify and characterize the target distribution.
6. Visualize the data.
7. Study the correlations between attributes.


In [ ]:
# Display the first few rows of the training data
print(train_df.head())

# Display information about the training data
print(train_df.info())

#Display just numerical data
df_num = train_df.select_dtypes(include = ['float64', 'int64'])
df_num.head()

In [ ]:
# Get a count of missing values for every column
missing_counts = train_df.isnull().sum()

# Filter to get only the columns that have one or more missing values
columns_with_missing = missing_counts[missing_counts > 0]

# Count how many columns are in the filtered list
num_cols_with_missing = len(columns_with_missing)

# Display the final count and the list of columns
print(f"Number of columns with missing values: {num_cols_with_missing}")
print("\n--- Columns With Missing Data ---")
print(columns_with_missing.sort_values(ascending=False))

**Types of Features**
The dataset contains a mix of feature types:

Numerical Features: These are represented by numbers (int64, float64) and can be either continuous or discrete.

Continuous: Can take any value within a range (e.g., LotArea, GrLivArea).

Discrete: Can only take specific integer values (e.g., BedroomAbvGr, GarageCars).

Categorical Features: These are represented by the object data type and contain a finite number of categories or labels.

Nominal: The categories have no intrinsic order (e.g., Street which is 'Pave' or 'Grvl', MSZoning).

Ordinal: The categories have a clear, meaningful order (e.g., ExterQual which includes 'Ex' (Excellent), 'Gd' (Good), 'TA' (Average), 'Fa' (Fair)). Although stored as text, these have a built-in ranking.

Identifier: The Id column is a unique identifier for each row and has no predictive value for the sale price.

**Missing Values**
As identified in the previous step, 19 columns have missing values. The ones with the most significant number of missing entries are shown in the code above.

For columns like PoolQC, Alley, and Fence, a missing value (NaN) likely means the house simply does not have that feature (e.g., no pool, no alley access).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import math

# --- Make sure train_df is loaded and SalePrice is log-transformed ---
# train_df['SalePrice'] = np.log1p(train_df['SalePrice'])

# Select all numerical columns except 'Id' and the target 'SalePrice'
numerical_features = train_df.select_dtypes(include=np.number).columns
numerical_features = numerical_features.drop(['Id', 'SalePrice'])

# Determine the grid size
n_features = len(numerical_features)
n_cols = 4
n_rows = math.ceil(n_features / n_cols)

# Create the subplot grid
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 3.5))
axes = axes.flatten() # Flatten the 2D array of axes for easy iteration

# Loop through the numerical features and create a scatter plot for each
for i, feature in enumerate(numerical_features):
    sns.scatterplot(data=train_df, x=feature, y='SalePrice', ax=axes[i], alpha=0.5)
    axes[i].set_title(f'{feature} vs. SalePrice', fontsize=10)
    axes[i].set_xlabel('') # Optional: remove x-labels for a cleaner look
    axes[i].set_ylabel('') # Optional: remove y-labels

# Hide any unused subplots
for i in range(n_features, len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
#creating chart for only numerical features
df_num.hist(figsize=(16, 20), bins=50, xlabelsize=8, ylabelsize=8);

In [ ]:
# Select all categorical columns
categorical_features = train_df.select_dtypes(include='object').columns

# Determine the grid size
n_features = len(categorical_features)
n_cols = 3
n_rows = math.ceil(n_features / n_cols)

# Create the subplot grid
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
axes = axes.flatten()

# Loop through the categorical features and create a box plot for each
for i, feature in enumerate(categorical_features):
    # Sort categories by median SalePrice for a cleaner plot
    order = train_df.groupby(feature)['SalePrice'].median().sort_values().index
    
    sns.boxplot(data=train_df, x=feature, y='SalePrice', ax=axes[i], order=order)
    axes[i].set_title(f'{feature} vs. SalePrice', fontsize=10)
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=45, horizontalalignment='right')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('')

# Hide any unused subplots
for i in range(n_features, len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
correlation_matrix = train_df.select_dtypes(include=np.number).corr()
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, cmap="coolwarm")
plt.title('Correlation Matrix of All Numeric Features')
plt.show()

In [ ]:
# Visualize outliers with a scatter plot
plt.figure(figsize=(10, 6))
sns.scatterplot(x=train_df['GrLivArea'], y=train_df['SalePrice'])
plt.title('GrLivArea vs. SalePrice (Log-Transformed)')
plt.xlabel('Gross Living Area (sq ft)')
plt.ylabel('Log SalePrice')
plt.show()



Useful Features: The features with the highest correlation to SalePrice will be the most useful for our model. Based on the heatmap, these include:

OverallQual: Overall material and finish quality.
GrLivArea: Above-grade living area.
GarageCars / GarageArea: Size of the garage.
TotalBsmtSF: Total square feet of the basement area.
These features have a strong, direct, and intuitive relationship with the value of a house. A bigger, higher-quality house with a large garage is expected to be more expensive.

Useless Features: Some features appear useless for prediction.
PoolQC, MiscFeature: With over 95% of their data missing, they offer very little information across the dataset.
Features with very low correlation to SalePrice (e.g., MoSold, BsmtFinSF2) are also less likely to be helpful.

There are two data points with very large GrLivArea (> 4000 sq ft) but surprisingly low SalePrice values. These points do not follow the general positive trend of the other data and are clear outliers. 

In [ ]:
# Visualize target distribution and apply log transformation
sns.set(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Original Distribution
sns.histplot(train_df['SalePrice'], kde=True, ax=axes[0], bins=50)
axes[0].set_title('Original Distribution of SalePrice')
axes[0].set_xlabel('Sale Price ($)')

# Log-Transformed Distribution
sns.histplot(np.log1p(train_df['SalePrice']), kde=True, ax=axes[1], color='green', bins=50)
axes[1].set_title('Log-Transformed Distribution of SalePrice')
axes[1].set_xlabel('Log(1 + Sale Price)')

plt.tight_layout()
plt.show()



Target Distribution
The distribution of our target variable, SalePrice, is positively skewed (or right-skewed).

Characterization: This means that while most houses fall into a lower-to-mid price range, there are a few very expensive houses that create a long "tail" to the right of the distribution.

Implication: This skew can be problematic for linear models, which perform best when variables (especially the target) are normally distributed.

Solution: To fix this, we apply a logarithmic transformation (np.log1p). This compresses the higher values, making the distribution much more symmetrical and "normal," as shown in the visualization below.

# 4. Prepare the Data for ML (15 pts)

Use the Scikit Learn data pipeline and preprocessing API. Write functions for any custom data transformations used that are not available in the Scikit Learn API.

Minimum expectations. Some of these may or may not be necessary for this task. If a step is not necessary, discuss why.

1. Clean the data.
2. Perform feature selection.
3. Perform feature engineering.
5. Standardize or normalize features.

Your data preparation pipeline does not need to use all of the features, but you must use at least one categorical feature, one ordinal feature, and one numeric feature. These can be raw features, or features that you create yourself.

In [ ]:
import numpy as np
import pandas as pd

# Path to the competition data
data_path = "../input/house-prices-advanced-regression-techniques/"

# Load the datasets
train_df = pd.read_csv(data_path + "train.csv")
test_df = pd.read_csv(data_path + "test.csv")

print("Datasets loaded successfully!")
# Permanently transform the SalePrice for future steps
print("Applying log transformation to the 'SalePrice' column for the model.")
train_df['SalePrice'] = np.log1p(train_df['SalePrice'])

Clean the Data

In [ ]:
# Display the original number of rows
print(f"Original number of training samples: {len(train_df)}")

# Remove the outliers
train_df = train_df.drop(train_df[(train_df['GrLivArea']>4000) & (train_df['SalePrice']<12.5)].index)

# Display the new number of rows
print(f"Number of training samples after removing outliers: {len(train_df)}")

Feature Selection
We have multiple features to choose from. A full pipeline would handle all of them, but for this assignment, I will select a few powerful ones based on my EDA to demonstrate the process. I am including the following data types:

Numeric Features: Overall Qual, GrLivArea, TotalBsmtSF, YearBuilt (High correlation with SalePrice).

Ordinal Feature: ExterQual, Kitchen Qual etc.

Categorical (Nominal) Feature: Neighborhood. This feature has no inherent order.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
import pandas as pd 

# 1. DEFINE THE FEATURE ENGINEERING FUNCTION

def create_features(df):
    """Creates new features and corrects data types."""
    # Ensure all required columns exist before calculation (e.g., '1stFlrSF', '2ndFlrSF' for TotalSF)
    # The function definition from your workbook is used here as it's more comprehensive.
    df['TotalSF'] = df['GrLivArea'] + df['TotalBsmtSF'] + df.get('1stFlrSF', 0) + df.get('2ndFlrSF', 0)
    df['TotalBath'] = (df['FullBath'] + 0.5 * df['HalfBath'] + 
                       df['BsmtFullBath'] + 0.5 * df['BsmtHalfBath'])
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df['AgeSinceRemod'] = df['YrSold'] - df['YearRemodAdd']
    df['QualitySF'] = df['OverallQual'] * df['TotalSF']
    df['YearBuilt_sq'] = df['YearBuilt']**2
    df['IsNew'] = (df['YrSold'] == df['YearBuilt']).astype(int)
    df['MSSubClass'] = df['MSSubClass'].astype(str)
    return df

# 2. APPLY FEATURE ENGINEERING
train_df_eng = create_features(train_df.copy())
test_df_eng = create_features(test_df.copy())
print("Feature engineering complete!")

# 3. SEPARATE TARGET AND FEATURES
X = train_df_eng.drop('SalePrice', axis=1)
y = train_df_eng['SalePrice']

# 4. DEFINE FEATURE LISTS AND CATEGORIES
numeric_features = [
    'GrLivArea', 'TotalBsmtSF', 'YearBuilt', 'TotalSF', 'TotalBath', 
    'HouseAge', 'AgeSinceRemod', 'QualitySF', 'YearBuilt_sq', 'IsNew'
]
ordinal_features = ['ExterQual']
nominal_features = ['Neighborhood', 'MSSubClass']

exter_qual_categories = ['Fa', 'TA', 'Gd', 'Ex']

# 5. CREATE INDIVIDUAL TRANSFORMER PIPELINES
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
ordinal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories=[exter_qual_categories]))
])
nominal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# 6. COMBINE TRANSFORMERS INTO A SINGLE PREPROCESSOR
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('ord', ordinal_transformer, ordinal_features),
        ('nom', nominal_transformer, nominal_features)
    ],
    remainder='drop' # Drop all other features
)

# 7. FIT and TRANSFORM THE TRAINING DATA
X_prepared = preprocessor.fit_transform(X)
print(f"Shape of the data after preprocessing: {X_prepared.shape}")

In [ ]:
from sklearn import set_config
# to visualize pipeline
# Set the display configuration to 'diagram'
set_config(display='diagram')

# Display the preprocessor object to see the visual diagram
preprocessor

# 5. Explore Different Models (15 pts)

Minimum expectations:

1. Train an unregularized linear model, a linear model with default parameters, and a decision tree.
2. Measure and compare their performance.
3. Analyze the most significant variables for each algorithm.
4. Analyze the types of errors the models make.


In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score, cross_val_predict, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error


# ---------- (A) Compare models (performance) ----------
cv = KFold(n_splits=10, shuffle=True, random_state=42)

lin_reg   = LinearRegression()
ridge_reg = Ridge(random_state=42)
tree_reg  = DecisionTreeRegressor(random_state=42)

def rmse_cv(estimator, X, y):
    scores = cross_val_score(estimator, X, y,
                             scoring="neg_root_mean_squared_error",
                             cv=cv)
    return -scores.mean(), scores.std()

lin_rmse_mean, lin_rmse_std     = rmse_cv(lin_reg,   X_prepared, y)
ridge_rmse_mean, ridge_rmse_std = rmse_cv(ridge_reg, X_prepared, y)
tree_rmse_mean, tree_rmse_std   = rmse_cv(tree_reg,  X_prepared, y)

print("=== Model Performance (10-fold CV, shuffled) ===")
print(f"Linear Regression  : RMSE = {lin_rmse_mean:.4f} ± {lin_rmse_std:.4f}")
print(f"Ridge Regression   : RMSE = {ridge_rmse_mean:.4f} ± {ridge_rmse_std:.4f}")
print(f"Decision Tree      : RMSE = {tree_rmse_mean:.4f} ± {tree_rmse_std:.4f}")

In [ ]:
# Fit on full prepared data so we can inspect coefficients/importances
lin_reg.fit(X_prepared, y)
ridge_reg.fit(X_prepared, y)
tree_reg.fit(X_prepared, y)

# ---------- (B) Feature names helper ----------
# Numeric + ordinal names
feature_names = []
feature_names.extend(numeric_features)
feature_names.extend(ordinal_features)

# One-hot names for the nominal block
try:
    ohe = preprocessor.named_transformers_["nom"].named_steps["encoder"]
    ohe_names = ohe.get_feature_names_out(nominal_features)
    feature_names.extend(list(ohe_names))
except Exception:
    # Fallback: just use generic names if anything is missing
    feature_names.extend([f"nom_{i}" for i in range(X_prepared.shape[1]-len(feature_names))])

# Safety: align lengths
if len(feature_names) != X_prepared.shape[1]:
    feature_names = [f"f{i}" for i in range(X_prepared.shape[1])]

# ---------- (C) (3) Most significant variables per algorithm ----------
def top_weights(coefs, names, k=10, label="coef"):
    df = pd.DataFrame({"feature": names, label: coefs})
    df["importance_abs"] = df[label].abs()
    return df.sort_values("importance_abs", ascending=False).head(k).drop(columns=["importance_abs"])

print("\n=== Top features: Linear Regression (by |coef|) ===")
display(top_weights(lin_reg.coef_, feature_names, k=10))

print("\n=== Top features: Ridge (by |coef|) ===")
display(top_weights(ridge_reg.coef_, feature_names, k=10))

print("\n=== Top features: Decision Tree (by importance) ===")
tree_imp = pd.DataFrame({
    "feature": feature_names,
    "importance": tree_reg.feature_importances_
}).sort_values("importance", ascending=False).head(10)
display(tree_imp)


In [ ]:
# ---------- (D) (4) Error analysis with cross-validated predictions ----------
# Use the best of the three as "analysis model" for residuals; pick the one with the lowest RMSE above
rmse_map = {
    "linear": lin_rmse_mean,
    "ridge": ridge_rmse_mean,
    "tree":  tree_rmse_mean
}
best_name = min(rmse_map, key=rmse_map.get)
best_model = {"linear": lin_reg, "ridge": ridge_reg, "tree": tree_reg}[best_name]

# Cross-validated predictions (out-of-fold) to avoid optimistic bias
y_pred_cv = cross_val_predict(best_model, X_prepared, y, cv=cv)
residuals = y - y_pred_cv
rmse = mean_squared_error(y, y_pred_cv, squared=False)
mae  = mean_absolute_error(y, y_pred_cv)

print(f"\n=== Error Analysis using {best_name.title()} (out-of-fold) ===")
print(f"RMSE: {rmse:.4f} | MAE: {mae:.4f}")

# Largest under/over-estimates
errs = pd.DataFrame({
    "y_true": y.values if hasattr(y, "values") else np.array(y),
    "y_pred": y_pred_cv,
})
errs["error"] = errs["y_pred"] - errs["y_true"]

print("\nTop 5 OVER-estimates (pred >> true):")
display(errs.sort_values("error", ascending=False).head(5))

print("Top 5 UNDER-estimates (pred << true):")
display(errs.sort_values("error").head(5))

Analysis of Model Prediction Errors 

**Linear Regression**
The standard Linear Regression model showed significant instability, as seen in its RMSE score. This model struggles because it's trying to fit a simple straight line to the complex, non-linear relationships in the housing data. 
Essentially, its reliance on purely linear relationships prevents it from accurately predicting prices at either end of the market spectrum.

**Ridge Regression**
The Ridge Regression model is the best one so far. Its behavior is similar to the standard linear model, but its errors are much more controlled and less extreme. The reason for this is regularization.
Regularization works by penalizing overly large coefficients, which makes the model more stable and less prone to being skewed by noise or multicollinearity (which our feature engineering introduced).
While it still has some trouble with the most extreme outliers, the overall magnitude of its over- and under-predictions is noticeably reduced, making it a far more consistent and reliable model.

**Decision Tree**
The tuned Decision Tree is better at capturing the non-linear patterns that the linear models miss. However, it has a different kind of error profile.
It lacks the smoothing effect of a linear model, so it struggles to produce nuanced predictions for unique, high-value properties that don't fit neatly into a category.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error

# --- (F) Calculate Training Error for each model ---
# The models are already fitted in the notebook cell above.
# If not, uncomment these lines:
# lin_reg.fit(X_prepared, y)
# ridge_reg.fit(X_prepared, y)
# tree_reg.fit(X_prepared, y)

# Linear Regression Training Error
y_pred_train_lin = lin_reg.predict(X_prepared)
lin_rmse_train = mean_squared_error(y, y_pred_train_lin, squared=False)

# Ridge Regression Training Error
y_pred_train_ridge = ridge_reg.predict(X_prepared)
ridge_rmse_train = mean_squared_error(y, y_pred_train_ridge, squared=False)

# Decision Tree Training Error
y_pred_train_tree = tree_reg.predict(X_prepared)
tree_rmse_train = mean_squared_error(y, y_pred_train_tree, squared=False)

# --- Create a DataFrame for plotting ---
error_data = {
    'Model': ['Linear Regression', 'Linear Regression', 
              'Ridge Regression', 'Ridge Regression', 
              'Decision Tree', 'Decision Tree'],
    'Error Type': ['Training', 'Cross-Validation',
                   'Training', 'Cross-Validation',
                   'Training', 'Cross-Validation'],
    'RMSE': [lin_rmse_train, lin_rmse_mean,
             ridge_rmse_train, ridge_rmse_mean,
             tree_rmse_train, tree_rmse_mean]
}
error_df = pd.DataFrame(error_data)

# --- Create the grouped bar plot ---
plt.figure(figsize=(12, 7))
sns.set_style("whitegrid")

ax = sns.barplot(x='Model', y='RMSE', hue='Error Type', data=error_df, palette='viridis')

# Add title and labels
plt.title('Training Error vs. Cross-Validation Error', fontsize=16)
plt.ylabel('Root Mean Squared Error (RMSE)', fontsize=12)
plt.xlabel('')
plt.xticks(fontsize=12)
plt.ylim(0, max(error_df['RMSE']) * 1.2)

# Add values on top of each bar
for p in ax.patches:
    ax.annotate(f'{p.get_height():.4f}', 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='center', 
                xytext=(0, 9), 
                textcoords='offset points',
                fontsize=11)

plt.show()

Linear and Ridge Regression
For these models, you can see that the Training Error and Cross-Validation Error bars are very close in height. This is a sign of a well-generalized model. It's not just "memorizing" the answers; it's learning the underlying patterns, so it performs consistently on both data it has seen and new data.

Decision Tree (Untuned)
For the Decision Tree, there is a huge gap between the two bars:
The Training Error is extremely low (close to zero). This is because a single, unconstrained Decision Tree is so flexible that it can create rules to perfectly fit almost every single data point in the training set. It essentially "memorizes" the practice test.
The Cross-Validation Error is much higher. When the "memorizing" model is shown new data during the "real test" (cross-validation), it performs poorly because it learned the noise and quirks of the training data, not the general patterns.

# 6.Fine-tune the Models (15 pts)

1. Fine-tune the hyperparameters for a linear model using RandomizedSearchCV.
2. Fine-tune the hyperparameters for a decision tree using GridSearchCV.


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform

# --- Ridge Regression Fine-Tuning (RandomizedSearchCV) ---
ridge_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('ridge', Ridge())
])

param_distributions = {
    'ridge__alpha': loguniform(1e-1, 1e2)
}

random_search = RandomizedSearchCV(ridge_pipeline, param_distributions,
                                   n_iter=50, cv=10,
                                   scoring='neg_mean_squared_error',
                                   random_state=42)
random_search.fit(X, y)

print("--- Ridge Regression Fine-Tuning (RandomizedSearchCV) ---")
print(f"Best hyperparameters found: {random_search.best_params_}")
best_rmse = np.sqrt(-random_search.best_score_)
print(f"Best cross-validation RMSE: {best_rmse:.4f}")

In [ ]:
from sklearn.model_selection import GridSearchCV

# --- Decision Tree Fine-Tuning (GridSearchCV) ---
tree_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('decisiontreeregressor', DecisionTreeRegressor(random_state=42))
])

param_grid = {
    'decisiontreeregressor__max_depth': [3, 5, 7, 10],
    'decisiontreeregressor__min_samples_leaf': [5, 10, 15, 20]
}

grid_search = GridSearchCV(tree_pipeline, param_grid, cv=50,
                           scoring='neg_mean_squared_error')
grid_search.fit(X, y)

print("\n--- Decision Tree Fine-Tuning (GridSearchCV) ---")
print(f"Best hyperparameters found: {grid_search.best_params_}")
best_rmse = np.sqrt(-grid_search.best_score_)
print(f"Best cross-validation RMSE: {best_rmse:.4f}")

**Learning Curve Analysis**

In [ ]:
from sklearn.model_selection import learning_curve
import matplotlib.pyplot as plt
import numpy as np

# Helper function to plot the learning curves
def plot_learning_curve(estimator, title, X, y, cv=None, n_jobs=None, train_sizes=np.linspace(0.1, 1.0, 10)):
    """
    Generate a simple plot of the test and training learning curve.
    """
    plt.figure(figsize=(8, 6))
    plt.title(title)
    plt.xlabel("Training examples")
    plt.ylabel("RMSE Score")
    
    train_sizes, train_scores, test_scores = learning_curve(
        estimator, X, y, cv=cv, n_jobs=n_jobs, train_sizes=train_sizes, scoring="neg_root_mean_squared_error"
    )
    
    train_scores_mean = -np.mean(train_scores, axis=1)
    test_scores_mean = -np.mean(test_scores, axis=1)

    plt.grid(True)
    plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Training score")
    plt.plot(train_sizes, test_scores_mean, 's-', color="g", label="Cross-validation score")
    plt.legend(loc="best")
    plt.show()

# --- Generate Learning Curves for Tuned Models ---

# 1. Tuned Ridge Regression
# The random_search object contains the full pipeline (preprocessor + model)
plot_learning_curve(random_search.best_estimator_, "Learning Curve - Tuned Ridge Regression", X, y, cv=cv, n_jobs=-1)

# 2. Tuned Decision Tree
# The grid_search object contains the full pipeline
plot_learning_curve(grid_search.best_estimator_, "Learning Curve - Tuned Decision Tree", X, y, cv=cv, n_jobs=-1)

Ridge Regression: You'll see that the training and validation curves are very close to each other and have converged to a low RMSE. This is the hallmark of a well-generalized model that is not overfitting. The minimum point on the red (CV) line is the optimal Alpha. Left (Low Alpha): Overfitting (High Variance). Right (High Alpha): Underfitting (High Bias).

Tuned Decision Tree: The validation curve will also converge close to the training curve, showing that tuning was successful in reducing overfitting. The final convergence point might be slightly higher than the Ridge model, confirming that Ridge is the better performer.

# 7. Summarize your results (15 pts)

1. Provide a 2-3 sentence summary of your solution for a non-technical audience of real estate executives.

2. Provide a technical summary of your solution for the ML team lead at a real estate investment company.

**Summary for Real Estate Executives**

We've developed a predictive model that accurately estimates the sale price of a house using its key features. By analyzing patterns in the Ames housing market, our system identifies that overall quality, living area, and neighborhood are the most critical drivers of a home's value. This tool allows us to make smarter, data-driven investment decisions by providing reliable price estimates for potential properties.


**Technical Summary for the ML Team Lead**

We framed this task as a supervised regression problem, evaluated on Root Mean Squared Logarithmic Error (RMSLE). Our final solution is a tuned Ridge Regression model integrated into a Scikit-Learn pipeline. The preprocessing pipeline first removes two major outliers from the GrLivArea feature, then handles missing data using median imputation for numeric features and most-frequent imputation for categorical ones; finally, it applies StandardScaler for numeric features and OneHotEncoder for categorical ones. After comparing Linear Regression, Ridge, and a Decision Tree, we fine-tuned the Ridge model's alpha hyperparameter using RandomizedSearchCV, which proved to be the top-performing model. For further improvement we can use the ensemble method, gradient boosted trees etc.

# 9. Submit your predictions to the competition

In [ ]:
# Select the best model and make predictions on the test set
final_model = random_search.best_estimator_
print("Making predictions on the test set...")
test_predictions_log = final_model.predict(test_df_eng)

# Inverse transform the log predictions to the original price scale
test_predictions = np.expm1(test_predictions_log)

# Create and save the submission file
submission_df = pd.read_csv(
    "/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv",
    index_col="Id"
).assign(SalePrice=test_predictions)

submission_df.to_csv("submission.csv")
print("Submission file created successfully!")
print(submission_df.head())

In [ ]:
%%bash

cat submission.csv | head -n 5